# 実習① 動画から肩の角度を測る

SAM-Body4Dが動画から取り出した「体の22点の位置」を使って、肩挙上角と肘屈曲角を計算し、グラフにします。

## このノートの使い方

- 灰色の箱を「セル」と呼びます。セルの中に書いてある文が「コード」（コンピューターへの手順書）です。
- セルの左の▶を押すと、そのセルが実行され、すぐ下に結果が出ます（Shift＋Enterでも同じ）。
- 上のセルから順に、全部押してください。下のセルは、上のセルで作ったものを使います。
- どこかを書き換えたら、そのセルから下を全部押し直してください。
- 結果が出ない・赤い文字が出たときは、メニューの［ランタイム］→［すべてのセルを実行］を押せば、上から順に押し直せます。

## 0. 最初の1行

セルの左の▶を押してみましょう。`print(…)`は、かっこの中を下に表示する命令です。すぐ下に「こんにちは」と出れば成功です。

In [ ]:
print('こんにちは')

## 1. 道具を読み込む

計算やグラフの道具は、最初は箱にしまわれています。`import`は「道具箱を開ける」命令です。
1行目は、グラフに日本語を出すための道具を入れる行です（先頭の`!`は、Colabに「これを入れて」と頼む合図）。

In [ ]:
!pip install -q matplotlib-fontja
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib_fontja
from scipy.signal import find_peaks
print('道具の準備ができました')

## 2. データを持ってくる

`FILE_ID = '…'`は、配られたIDに`FILE_ID`という名前を付けて取っておく行です。名前を付けて取っておいたものを「変数」と呼びます。
講師から配られたIDを、`'`と`'`の間に貼り付けてから▶を押してください。引用符`'`は消さないでください。

In [ ]:
FILE_ID = 'YOUR_FILE_ID_HERE'
!gdown {FILE_ID} -O rom_task_output.json

## 3. データを開く

持ってきたファイルを開き、中身に`data`という名前を付けて取っておきます。この2行は「ファイルを開いて、中身を`data`にしまう」というひとかたまりとして読めば十分です。

In [ ]:
with open('rom_task_output.json') as f:
    data = json.load(f)
print('データを開きました')

## 4. 中身の大きさを見る

`data`の中から、22点の位置（`joints`）と、1秒あたりのコマ数（`fps`）を取り出して、名前を付けます。
`data['joints']`のように、`[ ]`の中に名前を書くと、その名前の中身を取り出せます。`data['meta']['fps']`は、`meta`の中の`fps`です。

In [ ]:
joints = np.array(data['joints'])
fps = data['meta']['fps']
n_frames = len(joints)
print('コマ数', n_frames)
print('1秒あたりのコマ数', fps)
print('動画の長さ（秒）', n_frames / fps)

## 5. 1コマの22点をのぞく

`joints[150]`は150コマ目、`joints[150][17]`はその17番の点（右肩）です。番号は0から数えます。
出てくる3つの数は、横・上下・前後の位置（単位はメートル）です。`np.round(…, 3)`は小数第3位までに丸める命令です。

In [ ]:
print('右肩', np.round(joints[150][17], 3))
print('右肘', np.round(joints[150][19], 3))

## 6. 2本の線のなす角を出す関数

ゴニオメーターの代わりになる「関数」を作ります。関数は、入力を受け取って結果を返す手順のまとまりです。
`def 名前(入力):`で始まり、`return 結果`で終わります。このセルは中身を読み飛ばしてかまいません。
役割：2本の線のなす角を出す。入力：2本の線（`v1`・`v2`）。出力：角度（度）。
このセルを押しても何も表示されません。関数は、作っただけでは動かず、呼んだときに動きます。

In [ ]:
def angle_between_vectors(v1, v2):
    v1 = v1 / np.linalg.norm(v1)
    v2 = v2 / np.linalg.norm(v2)
    cos_angle = np.clip(np.dot(v1, v2), -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))

## 7. 肩挙上角の関数

ゴニオメーターの当て方をそのまま書きます。軸は肩、固定アームは体幹の線（首→骨盤）、移動アームは上腕の線（肩→肘）です。
役割：1コマの肩挙上角を出す。入力：1コマの22点と、右か左か。出力：角度（度）。

この角度は、屈曲と外転を区別しない「撮った面の中での挙上角」です。真横から撮れば屈曲、正面から撮れば外転として読みます。

In [ ]:
def compute_shoulder_elevation(frame, side='right'):
    # 1. どの点を使うか（番号は0から数える）
    if side == 'right':
        shoulder, elbow = frame[17], frame[19]
    else:
        shoulder, elbow = frame[16], frame[18]
    neck, pelvis = frame[12], frame[0]
    # 2. 2本の線を作る
    upper_arm = elbow - shoulder   # 移動アーム
    trunk = pelvis - neck          # 固定アーム
    # 3. なす角を返す
    return angle_between_vectors(upper_arm, trunk)

## 8. 1コマだけ測る

作った関数を呼んでみます。`関数の名前(入力)`と書くと、関数が動いて結果が返ってきます。
150コマ目（右肩がいちばん上がったコマ）を入れます。`round(…, 1)`は小数第1位までに丸める命令です。

In [ ]:
angle = compute_shoulder_elevation(joints[150], 'right')
print('150コマ目の肩挙上角', round(angle, 1))

## 9. 全コマを測る

同じ測定を、175コマぶん繰り返します。`for i in range(n_frames):`は「`i`を0、1、2…と変えながら、下の字下げした行を繰り返す」という意味です。
測った角度は、空の入れ物（`[]`）に1つずつ足していきます（`append`）。

やってみよう：`side = 'right'`を`side = 'left'`に変えて、このセルから下を全部押し直すと、左肩を測れます。

In [ ]:
side = 'right'
shoulder_elev = []
for i in range(n_frames):
    a = compute_shoulder_elevation(joints[i], side)
    shoulder_elev.append(a)
shoulder_elev = np.array(shoulder_elev)
print('最大', round(shoulder_elev.max(), 1))
print('最小', round(shoulder_elev.min(), 1))

## 10. 時系列のグラフを描く

横軸を時間、縦軸を肩挙上角にして、線で結びます。
`plt.plot(横, 縦)`が線を引く命令、`plt.xlabel`・`plt.ylabel`が軸の名前、`plt.show()`が表示です。

In [ ]:
time = np.arange(n_frames) / fps
plt.plot(time, shoulder_elev)
plt.xlabel('時間 [秒]')
plt.ylabel('肩挙上角 [度]')
plt.show()

## 11. 肘屈曲角も測る

肘も同じ形です。軸は肘、2本の線は肘→肩と肘→手首。2本の線のなす角（伸ばすと180°）を180°から引くと、臨床の屈曲角（伸ばすと0°）になります。
このセルは、7〜9と同じ形（関数を作る → 全コマで繰り返す）を1つにまとめたものです。

In [ ]:
def compute_elbow_flexion(frame, side='right'):
    if side == 'right':
        shoulder, elbow, wrist = frame[17], frame[19], frame[21]
    else:
        shoulder, elbow, wrist = frame[16], frame[18], frame[20]
    inner = angle_between_vectors(shoulder - elbow, wrist - elbow)
    return 180.0 - inner

elbow_flex = []
for i in range(n_frames):
    elbow_flex.append(compute_elbow_flexion(joints[i], side))
elbow_flex = np.array(elbow_flex)
print('肘屈曲の最大', round(elbow_flex.max(), 1))

## 12. 挙上のピークを数える

この動画では、同じ挙上が何度も繰り返されています。90°を超えた山を1回の挙上として数え、毎回のピークを並べます。
`find_peaks`は山の位置を探す道具です。答えを2つ返しますが、使うのは1つ目だけなので、2つ目は`_`で受けて捨てます。

In [ ]:
peaks, _ = find_peaks(shoulder_elev, height=90, prominence=30)
peak_values = shoulder_elev[peaks]
print('挙上の回数', len(peaks))
print('毎回のピーク', np.round(peak_values, 1))

## 13. ピークのばらつきを数にする

平均と標準偏差（SD）を出します。同じ人の同じ挙上を繰り返したときのSDは、測定誤差（SEM）の目安になります。
最小可検変化量 $\mathrm{MDC}_{95} = 1.96 \times \sqrt{2} \times \mathrm{SEM}$ を、式のとおりに1行で書きます。

In [ ]:
mean = peak_values.mean()
sd = peak_values.std(ddof=1)
mdc95 = 1.96 * np.sqrt(2) * sd
print('平均', round(mean, 1))
print('SD', round(sd, 1))
print('MDC95', round(mdc95, 1))

## 14. 実習の問い

グラフと数を見ながら、気づいたことをメモしてください。

1. 時系列のグラフで、挙上は毎回同じ高さまで上がっているか。途中で止まった回はあるか。
2. 右と左（9のセルで`side`を変える）で、最大値はどれだけ違うか。その差は、MDC95と比べて大きいか。
3. この動画は真横から撮られている。肩挙上角は屈曲・外転のどちらとして読むべきか。
4. 次回同じ撮り方で測って「良くなった」と言うには、ピークの平均が何度変わればよいか。